In [17]:
import torch

# torch.cumsum
A = torch.tensor([1, 1, 1, 1])

B = torch.cumsum(A, dim=0)
print(B)  # tensor([1, 2, 3, 4])

import numpy as np

a = np.array([1, 2, 3, 4])
b = np.cumsum(a, axis=0)
print(b)  # [1 3 6 10]

tensor([1, 2, 3, 4])
[ 1  3  6 10]


In [4]:
def EnvConfig_v1(envName: str):
    print(f"Using environment configuration for: {envName}")
    return {
        # System / episode
        "num_users": 10,
        "T": 10,                    # episode length (steps)
        "sys_tau": 4,             # system latency budget (s)
        "Gmax": 5e9,                # FLOPS budget per step
        "Mmax": 48,                 # memory budget (arbitrary units)

        # Penalty weights
        "lambda_qos": 0.5,
        "lambda_latency": 0.5,
        "lambda_mem": 1.0,
        "lambda_flops": 1.0,
        "lambda_price": 1.0,        # NEW: pricing component weight

        # Compute / memory capacity
        "PVM": 1e12,                # 1 TFLOPS (bytes/sec when dividing flops? here used as FLOP/s)
        "Rmem": 2.304e12,           # memory bandwidth (bytes/s)

        # Denoise step bounds
        "max_denoise_steps": 15,
        "min_denoise_steps": 3,

        # Piecewise quadratic memory model (NEW)
        # Lower resolution: a1*px^2 + a2*px + a3
        "mem_a1": 1e-8,
        "mem_a2": 2e-4,
        "mem_a3": 1.0,
        # Mid resolution: constant
        "mem_const": 12.0,
        # High resolution: b1*px^2 + b2*px + b3
        "mem_b1": 5e-9,
        "mem_b2": 1e-4,
        "mem_b3": 8.0,
        "mem_threshold_low": 1024**2,    # pixels
        "mem_threshold_high": 1792**2,   # pixels
        # Old linear model (kept for backward compatibility)
        # "c1": 3.81e-6,
        # "c2": 4.86,

        # Workload model
        "base_image_size": 1024 * 1024,  # 1 MiB reference
        "base_resolution": 512 * 512,    # reference resolution in pixels (NEW)
        "GE0": 1e8,       # base FLOPS encoder
        "GD0": 1e8,       # base FLOPS decoder
        "G_eps": 1e8,     # FLOPS per denoise step
        "G_prompt": 1e7,  # FLOPS for prompt processing

        # Pricing model (NEW: explicit FLOPs pricing)
        "lambda_m": 1e-8,   # memory pricing coefficient
        "lambda_g": 5e-10,  # FLOPS pricing coefficient
        "lambda_c": 2.5e-7, # communication pricing coefficient

        # Latency model (NEW: denoising/overhead latency)
        "t_ldm_overhead": 0.005,  # fixed LDM overhead (s)
        "t_per_denoise": 0.0005,  # time per denoise step (s)

        # Wireless link / geometry
        "sp_pos": np.array([0.0, 0.0, 50.0]),
        "h0": 1.42e-4,
        "path_loss": 2.0,
        "bandwidth": 1e6,           # Hz
        "noise_power": 4.0e-21,     # W/Hz
        "upload_power": 0.0501,     # W
        "download_power": 0.5012,   # W

        # Reward bonus
        "psi": 100,

        # QoS target (lower PIQUE is better)
        "qos_required": 30,
    }

In [7]:
import numpy as np

config = EnvConfig_v1("GAIServiceEnv")

class User:
    def __init__(self, user_id: int, config: dict, rng: np.random.Generator):
        self.user_id = user_id
        self.config = config
        self.rng = rng
        self.reset(config)

    def reset(self, config=None):
        if config is None:
            config = self.config
        # Random position on ground plane (x,y), z=0
        self.position = self.rng.uniform(-500.0, 500.0, size=2)

        # Treat sizes as KB for realism, convert to bytes
        self.image_size = float(self.rng.uniform(100, 350) * 1024.0)  # 100–1000 KB
        self.prompt_size = float(self.rng.uniform(1, 10) * 1024.0)   # 10–100 KB

        self.direction = float(self.rng.uniform(0.0, 2.0 * np.pi))
        self.qos_required = config["qos_required"]
        self.mobility_speed = float(self.rng.uniform(0.5, 2.0))  # m/step
        self.mobility_angle = self.direction

    def update_position(self):
        dx = self.mobility_speed * np.cos(self.mobility_angle)
        dy = self.mobility_speed * np.sin(self.mobility_angle)
        self.position += np.array([dx, dy], dtype=np.float64)
        self.mobility_angle += float(self.rng.uniform(-0.1, 0.1))


users = [User(i, config, np.random.default_rng()) for i in range(config["num_users"])]

users

Using environment configuration for: GAIServiceEnv


In [11]:
len(users)

10

In [9]:
observations = []
for user in users:
    obs = np.array([
        user.position[0],
        user.position[1],
        user.image_size,
        user.prompt_size,
        user.qos_required
    ], dtype=np.float32)
    observations.append(obs)
observations = np.concatenate(observations)
observations.shape

(50,)

In [16]:
actions = np.random.uniform(0.0, 1.0, size=(2 * config["num_users"],)).astype(np.float32)
actions.shape

def map_action_to_decisions(action: np.ndarray, config: dict):
    num_users = config["num_users"]
    denoise_steps = []
    upload_resolutions = []
    for i in range(num_users):
        # Denoise steps
        a_ds = action[i]
        ds = int(config["min_denoise_steps"] + a_ds * (config["max_denoise_steps"] - config["min_denoise_steps"]))
        denoise_steps.append(ds)

        # Upload resolution
        a_res = action[num_users + i]
        res = int(256 + a_res * (2560 - 256))  # from 256 to 2560 pixels
        upload_resolutions.append(res)

    return denoise_steps, upload_resolutions

denoise_steps, upload_resolutions = map_action_to_decisions(actions, config)
denoise_steps, upload_resolutions

([14, 9, 9, 8, 14, 11, 11, 13, 14, 14],
 [490, 1476, 2241, 2457, 1749, 1965, 2428, 405, 1190, 1188])

In [13]:
user_positions = [user.position for user in users]
user_positions = np.array(user_positions)
user_positions

array([[-442.48295281,  135.30776519],
       [-225.29283564, -343.19093148],
       [-489.21949259, -399.94771498],
       [-102.35742655,  236.70421326],
       [-251.38267621, -364.30678579],
       [ -91.8648463 , -389.68026311],
       [ -20.45531838, -146.4601905 ],
       [  36.61747989,  435.57688732],
       [  29.54779094,  323.38317132],
       [ 289.07489883, -458.03824473]])

In [15]:
user_image_sizes = [user.image_size for user in users]
user_image_sizes = np.array(user_image_sizes)
user_prompt_sizes = [user.prompt_size for user in users]
user_prompt_sizes = np.array(user_prompt_sizes)
user_quos_required = [user.qos_required for user in users]
print(user_image_sizes, user_prompt_sizes, user_quos_required, sep="\n\n")

[102756.56091537 282980.56865098 257054.91406683 259792.71661166
 290192.88302303 132246.36680323 133776.17189056 240426.52178839
 291550.74297124 106455.59612206]

[2121.50270303 5556.41959704 7804.62293571 1940.19143279 3452.74593612
 8279.9322078  5141.69472488 6674.06087497 9169.66174561 2638.7233056 ]

[30, 30, 30, 30, 30, 30, 30, 30, 30, 30]
